# Samplers: Sampling and Signal Reconstruction

**Learning Goals**
- Understand what sampling is and why it is needed for digital control.
- Understand the Nyquist–Shannon sampling theorem.
- Understand what a zero-order hold (ZOH) does.
- Understand what a first-order hold (FOH) does.
- See how sampling rate and hold type affect the block-on-ice system.

---

In [1]:
%pip install -q ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from scipy.fft import fft, fftfreq

print("Libraries loaded.")

Libraries loaded.


---

## What is sampling?

A **sampler** converts a continuous-time signal $x(t)$ into a sequence of discrete values:

$$x_k = x(k T_s), \quad k = 0, 1, 2, \dots$$

where $T_s$ is the **sampling period** and $f_s = 1/T_s$ is the **sampling frequency**.

Think of a sampler as a switch that closes briefly every $T_s$ seconds to take a snapshot of the signal. Between snapshots, the sampler sees nothing.

### The Nyquist–Shannon Theorem

To faithfully reconstruct a continuous signal from its samples, the sampling frequency must satisfy:

$$f_s > 2 f_{\text{max}}$$

where $f_{\text{max}}$ is the highest frequency in the signal. The frequency $f_s/2$ is called the **Nyquist frequency**.

#### Example

If a signal contains a 100 Hz vibration, what is the minimum sampling rate needed to capture it?

At least $f_s > 2 \times 100 = 200$ Hz. Anything less and the sampled signal will be **aliased**: it will look like a different, lower frequency.

In [3]:
f1_fixed = 50
f2_fixed = 120
fmax = max(f1_fixed, f2_fixed)

def interactive_sampling(fs_good, fs_marginal, fs_bad):
    t_cont = np.linspace(0, 0.1, 2000)
    x_cont = np.sin(2 * np.pi * f1_fixed * t_cont) + 0.5 * np.sin(2 * np.pi * f2_fixed * t_cont)

    fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

    cases = [
        (fs_good, f'$f_s = {fs_good:.0f}$ Hz; $f_s \\gg 2f_{{\\max}}$ (GOOD)'),
        (fs_marginal, f'$f_s = {fs_marginal:.0f}$ Hz; $f_s \\approx 2f_{{\\max}}$ (marginal)'),
        (fs_bad, f'$f_s = {fs_bad:.0f}$ Hz; $f_s < 2f_{{\\max}}$ (ALIASING!)'),
    ]

    for ax, (fs, label) in zip(axes, cases):
        Ts = 1.0 / fs
        t_s = np.arange(0, 0.1, Ts)
        x_s = np.sin(2 * np.pi * f1_fixed * t_s) + 0.5 * np.sin(2 * np.pi * f2_fixed * t_s)
        ax.plot(t_cont * 1000, x_cont, 'b-', linewidth=1, alpha=0.5, label='Continuous')
        ax.stem(t_s * 1000, x_s, linefmt='r-', markerfmt='ro', basefmt=' ',
                label=f'Samples ($f_s = {fs:.0f}$ Hz)')
        ax.axhline(0, color='k', alpha=0.2)
        ax.axvline(1000 / (2 * fmax), color='g', ls=':', alpha=0.6, label=f'Nyquist period = {1000/(2*fmax):.1f} ms')
        ax.set(ylabel='Amplitude', title=label)
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

    axes[-1].set(xlabel='Time (ms)')
    fig.tight_layout()
    plt.show()
    print(f'Signal: $f_1 = {f1_fixed}$ Hz, $f_2 = {f2_fixed}$ Hz  |  $f_{{\\max}} = {fmax}$ Hz  |  Nyquist rate = {2*fmax} Hz')

fs_bad_s = widgets.FloatSlider(min=10, max=2*fmax-1, step=5, value=80,
                                description='Bad $f_s$:', style={'description_width': 'initial'})
fs_marg_s = widgets.FloatSlider(min=2*fmax+10, max=5*fmax, step=10, value=300,
                                  description='Marginal $f_s$:', style={'description_width': 'initial'})
fs_good_s = widgets.FloatSlider(min=5*fmax+10, max=2000, step=10, value=1000,
                                  description='Good $f_s$:', style={'description_width': 'initial'})

run_btn_s = widgets.Button(description='Run', button_style='primary')
out_s = widgets.Output()

def on_run_s(b):
    with out_s:
        clear_output(wait=True)
        interactive_sampling(fs_good_s.value, fs_marg_s.value, fs_bad_s.value)

run_btn_s.on_click(on_run_s)
display(widgets.VBox([
    widgets.HBox([fs_bad_s, fs_marg_s, fs_good_s]),
    run_btn_s, out_s
]))
print("Signal fixed at 50 Hz + 120 Hz. Adjust the three sampling rates, then click Run.")

Signal fixed at 50 Hz + 120 Hz. Adjust the three sampling rates, then click Run.


---

## What is a zero-order hold?

Once we have samples, a computer can process them. But the physical world is continuous. To send a command back to the system, we need to **reconstruct** a continuous signal from the discrete samples. A **hold** device does this.

The simplest hold is the **zero-order hold (ZOH)**. It holds each sample value constant until the next sample arrives:

$$x_{\text{ZOH}}(t) = x(k T_s), \quad k T_s \leq t < (k+1) T_s$$

This is exactly what a **digital-to-analog converter (DAC)** does: it outputs a constant voltage until the next sample updates it.

### What about a first-order hold (FOH)?

A **first-order hold (FOH)** linearly interpolates between samples instead of holding constant. This gives a smoother reconstruction, but it is more complex and less common in practice. We will focus on the ZOH.

In [4]:
def interactive_holds(freq=80, fs=200):
    t_cont = np.linspace(0, 3 / freq, 2000)
    x_cont = np.sin(2 * np.pi * freq * t_cont)

    Ts = 1.0 / fs
    t_s = np.arange(0, t_cont[-1], Ts)
    x_s = np.sin(2 * np.pi * freq * t_s)

    t_zoh = t_cont.copy()
    x_zoh = np.zeros_like(t_zoh)
    for k in range(len(t_s) - 1):
        mask = (t_zoh >= t_s[k]) & (t_zoh < t_s[k + 1])
        x_zoh[mask] = x_s[k]
    x_zoh[t_zoh >= t_s[-1]] = x_s[-1]

    t_foh = t_zoh.copy()
    x_foh = np.zeros_like(t_foh)
    for k in range(len(t_s) - 1):
        mask = (t_foh >= t_s[k]) & (t_foh < t_s[k + 1])
        x_foh[mask] = x_s[k] + (t_foh[mask] - t_s[k]) / Ts * (x_s[k + 1] - x_s[k])
    x_foh[t_foh >= t_s[-1]] = x_s[-1]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    ax1.plot(t_cont * 1000, x_cont, 'b-', linewidth=2, alpha=0.7, label='Original')
    ax1.step(t_s * 1000, x_s, 'r-', where='post', linewidth=2, label='ZOH')
    ax1.stem(t_s * 1000, x_s, linefmt='k-', markerfmt='ko', basefmt=' ', label='Samples')
    ax1.set(ylabel='Amplitude', title=f'ZOH — $f = {freq}$ Hz, $f_s = {fs}$ Hz')
    ax1.legend(fontsize=8)
    ax1.grid(alpha=0.3)

    ax2.plot(t_cont * 1000, x_cont, 'b-', linewidth=2, alpha=0.7, label='Original')
    ax2.plot(t_foh * 1000, x_foh, 'g-', linewidth=2, label='FOH')
    ax2.stem(t_s * 1000, x_s, linefmt='k-', markerfmt='ko', basefmt=' ', label='Samples')
    ax2.set(xlabel='Time (ms)', ylabel='Amplitude', title=f'FOH — $f = {freq}$ Hz, $f_s = {fs}$ Hz')
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)

    fig.tight_layout()
    plt.show()

freq_h = widgets.FloatSlider(min=20, max=500, step=5, value=80,
                              description='Signal $f$ (Hz):', style={'description_width': 'initial'})
fs_h = widgets.FloatSlider(min=20, max=2000, step=10, value=200,
                            description='$f_s$ (Hz):', style={'description_width': 'initial'})

run_btn_h = widgets.Button(description='Run', button_style='primary')
out_h = widgets.Output()

def on_run_h(b):
    with out_h:
        clear_output(wait=True)
        interactive_holds(freq_h.value, fs_h.value)

run_btn_h.on_click(on_run_h)
display(widgets.VBox([widgets.HBox([freq_h, fs_h]), run_btn_h, out_h]))
print("Adjust signal frequency and sampling rate, then click Run.")

Adjust signal frequency and sampling rate, then click Run.


---

## Practical example: sampling the block on ice

Let us revisit the block-on-ice system from the systems notebook. The proportional controller applies force:

$$F(t) = K_p \, (r - x(t))$$

Now imagine the controller is implemented on a **digital computer**. The computer can only measure the block's position and update the force at discrete sampling instants. Between samples, the force is held constant by a **zero-order hold**.

The simulation below runs the same physics but the controller only sees the block's position at intervals of $T_s = 1 / f_s$.

Adjust the sampling rate and see how it affects the closed-loop response.

In [5]:
def sample_block_on_ice(Kp=2.0, friction=0.5, mass=10.0, fs=60, hold='ZOH'):
    setpoint = 10.0
    tau = mass / (friction + Kp)
    x_ss = Kp / (friction + Kp) * setpoint

    t_span = 6.0
    dt_cont = 1 / 10000
    t_cont = np.arange(0, t_span, dt_cont)
    x_cont = x_ss * (1 - np.exp(-t_cont / tau))

    T_s = 1.0 / fs
    t_s = np.arange(0, t_span, T_s)
    x_s = x_ss * (1 - np.exp(-t_s / tau))

    t_rec = t_cont.copy()
    if hold == 'ZOH':
        x_rec = np.zeros_like(t_rec)
        for k in range(len(t_s) - 1):
            mask = (t_rec >= t_s[k]) & (t_rec < t_s[k + 1])
            x_rec[mask] = x_s[k]
        x_rec[t_rec >= t_s[-1]] = x_s[-1]
    else:
        x_rec = np.zeros_like(t_rec)
        for k in range(len(t_s) - 1):
            mask = (t_rec >= t_s[k]) & (t_rec < t_s[k + 1])
            x_rec[mask] = x_s[k] + (t_rec[mask] - t_s[k]) / T_s * (x_s[k + 1] - x_s[k])
        x_rec[t_rec >= t_s[-1]] = x_s[-1]

    error = x_rec - x_cont

    fig = plt.figure(figsize=(14, 9))
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1])

    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(t_cont, x_cont, 'b-', linewidth=2, alpha=0.6, label='Continuous')
    ax1.step(t_s, x_s, 'r-', where='post', linewidth=1.5, alpha=0.8, label=f'{hold} reconstructed')
    ax1.stem(t_s, x_s, linefmt='k-', markerfmt='ko', basefmt=' ', label='Samples')
    ax1.axhline(setpoint, color='k', ls='--', alpha=0.3)
    ax1.set(xlabel='Time (s)', ylabel='Position (m)', title=f'{hold} Reconstruction — $f_s = {fs}$ Hz')
    ax1.legend(fontsize=8)
    ax1.grid(alpha=0.3)

    ax2 = fig.add_subplot(gs[1, 0])
    zoom_end = min(1.5, t_span)
    mask = t_cont <= zoom_end
    ax2.plot(t_cont[mask], x_cont[mask], 'b-', linewidth=2, alpha=0.6, label='Continuous')
    mask_s = t_s <= zoom_end
    if hold == 'ZOH':
        ax2.step(t_s[mask_s], x_s[mask_s], 'r-', where='post', linewidth=2, label=f'{hold}')
    else:
        t_foh = t_cont[t_cont <= zoom_end]
        x_foh = np.interp(t_foh, t_s, x_s)
        ax2.plot(t_foh, x_foh, 'r-', linewidth=2, label=f'{hold}')
    ax2.stem(t_s[mask_s], x_s[mask_s], linefmt='k-', markerfmt='ko', basefmt=' ')
    for ts in t_s[mask_s]:
        ax2.axvline(ts, color='k', alpha=0.1, linewidth=0.5)
    ax2.axhline(setpoint, color='k', ls='--', alpha=0.3)
    ax2.set(xlabel='Time (s)', ylabel='Position (m)', title=f'Zoom: First {zoom_end:.1f} s')
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)

    ax3 = fig.add_subplot(gs[1, 1])
    ax3.plot(t_cont, error, 'm-', linewidth=1.5)
    ax3.axhline(0, color='k', alpha=0.3)
    ax3.set(xlabel='Time (s)', ylabel='Error (m)', title=f'Reconstruction Error (RMS = {np.sqrt(np.mean(error**2)):.4f} m)')
    ax3.grid(alpha=0.3)

    ax4 = fig.add_subplot(gs[2, :])
    ax4.axis('off')

    fig.tight_layout()
    plt.show()

Kp_s = widgets.FloatSlider(min=0.1, max=10.0, step=0.1, value=2.0,
                           description='$K_p$:', style={'description_width': 'initial'})
fric_s = widgets.FloatSlider(min=0.0, max=2.0, step=0.05, value=0.5,
                              description='$b$:', style={'description_width': 'initial'})
mass_s = widgets.FloatSlider(min=1.0, max=20.0, step=0.5, value=10.0,
                              description='$m$:', style={'description_width': 'initial'})
fs_s = widgets.FloatSlider(min=5, max=50, step=1, value=5,
                           description='$f_s$ (Hz):', style={'description_width': 'initial'})
hold_s = widgets.Dropdown(options=['ZOH', 'FOH'], value='ZOH',
                          description='Hold:', style={'description_width': 'initial'})

run_btn = widgets.Button(description='Run', button_style='primary')
out = widgets.Output()

def on_run(b):
    with out:
        clear_output(wait=True)
        sample_block_on_ice(Kp_s.value, fric_s.value, mass_s.value,
                            fs_s.value, hold_s.value)

run_btn.on_click(on_run)
display(widgets.VBox([
    widgets.HBox([Kp_s, fric_s, mass_s]),
    widgets.HBox([fs_s, hold_s]),
    run_btn, out
]))
print("Adjust system parameters, sampling rate, and hold type, then click Run.")

Adjust system parameters, sampling rate, and hold type, then click Run.
